# Assignment 7 - Mode Collapse in a DCGAN
Run: Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

In [ ]:
import torch, torch.nn as nn, torchvision, torchvision.transforms as T
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", dev)
NZ, EPOCHS, BS = 100, 15, 128

data = torchvision.datasets.MNIST("./data", download=True,
        transform=T.Compose([T.ToTensor(), T.Normalize(0.5, 0.5)]))
loader = torch.utils.data.DataLoader(data, batch_size=BS, shuffle=True, drop_last=True)

In [ ]:
def make_G(ngf, bn):
    n = (lambda c: nn.BatchNorm2d(c)) if bn else (lambda c: nn.Identity())
    return nn.Sequential(
        nn.ConvTranspose2d(NZ, ngf*4, 7, 1, 0, bias=False), n(ngf*4), nn.ReLU(True),
        nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False), n(ngf*2), nn.ReLU(True),
        nn.ConvTranspose2d(ngf*2, 1, 4, 2, 1), nn.Tanh()).to(dev)

def make_D():
    return nn.Sequential(
        nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2, True),
        nn.Conv2d(64, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
        nn.Conv2d(128, 1, 7, 1, 0), nn.Flatten()).to(dev)

def diversity(x):  # mean pairwise distance; near 0 = collapse
    x = x.reshape(x.size(0), -1)
    return torch.cdist(x, x).mean().item()

def show(path, title):
    img = plt.imread(path)
    plt.figure(figsize=(7,7)); plt.imshow(img, cmap="gray"); plt.axis("off"); plt.title(title); plt.show()

def train(name, ngf, bn, lr_g, lr_d, real_label):
    torch.manual_seed(0)
    G, D = make_G(ngf, bn), make_D()
    oG = torch.optim.Adam(G.parameters(), lr_g, betas=(0.5, 0.999))
    oD = torch.optim.Adam(D.parameters(), lr_d, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()
    fixed = torch.randn(64, NZ, 1, 1, device=dev)
    for ep in range(1, EPOCHS + 1):
        for real, _ in loader:
            real = real.to(dev)
            fake = G(torch.randn(BS, NZ, 1, 1, device=dev))
            lossD = bce(D(real), torch.full((BS, 1), real_label, device=dev)) + \
                    bce(D(fake.detach()), torch.zeros(BS, 1, device=dev))
            oD.zero_grad(); lossD.backward(); oD.step()
            lossG = bce(D(fake), torch.ones(BS, 1, device=dev))
            oG.zero_grad(); lossG.backward(); oG.step()
        with torch.no_grad():
            s = G(fixed)
        print(f"[{name}] epoch {ep}/{EPOCHS} lossD={lossD.item():.3f} lossG={lossG.item():.3f} diversity={diversity(s):.2f}")
    save_image(s * 0.5 + 0.5, f"{name}.png", nrow=8, padding=2)
    return diversity(s)

real_div = diversity(next(iter(loader))[0][:64].to(dev))
print("Real data diversity:", round(real_div, 2))

## 1) Broken model
Tiny generator + no BatchNorm in G + learning rate 0.01

In [ ]:
b = train("broken_collapsed", ngf=8, bn=False, lr_g=0.01, lr_d=0.01, real_label=1.0)
show("broken_collapsed.png", "BROKEN / COLLAPSED output  -> screenshot this")

## 2) Fixed model
Label smoothing (real = 0.9) + different learning rates (G = 2e-4, D = 1e-4)

In [ ]:
f = train("fixed_improved", ngf=64, bn=True, lr_g=2e-4, lr_d=1e-4, real_label=0.9)
show("fixed_improved.png", "IMPROVED output  -> screenshot this")

## 3) Comparison

In [ ]:
print(f"Diversity  real={real_div:.2f} | broken={b:.2f} | fixed={f:.2f}")
fig, ax = plt.subplots(1, 2, figsize=(12,6))
for a, p, t in zip(ax, ["broken_collapsed.png","fixed_improved.png"], ["Broken (collapsed)","Fixed (improved)"]):
    a.imshow(plt.imread(p), cmap="gray"); a.set_title(t); a.axis("off")
plt.show()